In [1]:
import json
import os
from pydantic import BaseModel
from typing import List
from dotenv import load_dotenv
from groq import Groq

load_dotenv(override=True)

client = Groq(api_key=os.getenv("GROQ_API_KEY"))

In [3]:
class JobRequirements(BaseModel):
    job_title: str
    required_skills: List[str]
    nice_to_have_skills: List[str]
    experience_years: int
    keywords: List[str]

def extract_job_requirements(job_description: str) -> dict:
    schema = JobRequirements.model_json_schema()
    
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "system",
                "content": f"""Extract structured requirements from a job description.
Return only a JSON object matching this schema: {schema}
Return only the JSON, no other text."""
            },
            {
                "role": "user",
                "content": job_description
            }
        ],
        response_format={"type": "json_object"}
    )
    
    return json.loads(response.choices[0].message.content)

In [4]:
test_jd = """
Data Analyst at a Logistics Company

We are looking for a Data Analyst with 3+ years of experience.

Required:
- Strong SQL skills
- Python for data analysis (pandas, numpy)
- Power BI or Tableau for dashboards
- Experience with logistics or supply chain data

Nice to have:
- dbt experience
- Azure or AWS
- Freight industry knowledge
"""

result = extract_job_requirements(test_jd)
print(json.dumps(result, indent=2))

{
  "job_title": "Data Analyst",
  "required_skills": [
    "SQL",
    "Python",
    "pandas",
    "numpy",
    "Power BI",
    "Tableau",
    "logistics data",
    "supply chain data"
  ],
  "nice_to_have_skills": [
    "dbt",
    "Azure",
    "AWS",
    "Freight industry knowledge"
  ],
  "experience_years": 3,
  "keywords": [
    "Data Analyst",
    "Logistics",
    "Data Analysis",
    "SQL",
    "Python",
    "Power BI",
    "Tableau"
  ]
}


In [5]:
class CVScore(BaseModel):
    match_score: int
    matched_keywords: List[str]
    missing_keywords: List[str]
    experience_match: bool
    summary: str

def score_cv(cv_text: str, requirements: dict) -> dict:
    schema = CVScore.model_json_schema()
    
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "system",
                "content": f"""You are an ATS scoring system. Compare a CV against job requirements.
Return only a JSON object matching this schema: {schema}

match_score is 0-100 representing how well the CV matches the requirements.
matched_keywords are keywords from requirements that appear in the CV.
missing_keywords are required keywords NOT found in the CV.
experience_match is true if the CV meets the required years of experience.
summary is 1-2 sentences explaining the score.

Return only the JSON, no other text."""
            },
            {
                "role": "user",
                "content": f"CV:\n{cv_text}\n\nRequirements:\n{json.dumps(requirements, indent=2)}"
            }
        ],
        response_format={"type": "json_object"}
    )
    
    return json.loads(response.choices[0].message.content)

In [6]:
test_cv = """
John Smith
Data Analyst | john@email.com

EXPERIENCE
Senior Data Analyst - ABC Corp (2021-Present, 3 years)
- Built SQL queries to analyse customer data
- Created Power BI dashboards for operations team
- Used Python and pandas for data cleaning

Junior Analyst - XYZ Ltd (2019-2021)
- Supported reporting using Excel and SQL

SKILLS
SQL, Python, pandas, Excel, Power BI, data visualisation

EDUCATION
BSc Statistics - University of Manchester (2019)
"""

score_result = score_cv(test_cv, result)
print(json.dumps(score_result, indent=2))

{
  "match_score": 70,
  "matched_keywords": [
    "Data Analyst",
    "SQL",
    "Python",
    "Power BI"
  ],
  "missing_keywords": [
    "Logistics",
    "Tableau",
    "numpy",
    "logistics data",
    "supply chain data"
  ],
  "experience_match": true,
  "summary": "This CV matches the Data Analyst job requirements well, with relevant skills and experience, but lacks some specific technical skills and logistics-related keywords. The candidate has the required 3 years of experience, but may need to learn or gain experience in areas like Tableau and logistics data analysis."
}


In [7]:
from minsearch import Index

documents = [
    {"title": "ATS Keyword Matching", "category": "ats_basics", "content": "ATS systems scan CVs for exact keyword matches from the job description. If your CV uses 'managed projects' but the JD says 'project management', the ATS may not match them. Mirror the exact phrases used in the job description wherever possible."},
    {"title": "CV Formatting for ATS", "category": "formatting", "content": "ATS systems struggle with tables, columns, headers and footers, text boxes, and graphics. Use a single column layout with standard section headings like Experience, Education, and Skills. Save as a plain .docx or PDF without complex formatting."},
    {"title": "Skills Section Optimisation", "category": "keywords", "content": "Include a dedicated Skills section that lists your technical and soft skills as individual keywords. This is the easiest section for ATS to parse. Match skills terminology directly from the job description."},
    {"title": "Tailoring CV to Job Description", "category": "tailoring", "content": "A generic CV performs poorly in ATS screening. For each application, identify the top 5-10 keywords in the job description and ensure they appear naturally in your CV. Focus on the required qualifications section as it carries the most ATS weight."},
    {"title": "Action Verbs and Quantification", "category": "content_quality", "content": "Use strong action verbs at the start of each bullet point: led, built, improved, reduced, managed. Quantify achievements wherever possible: increased sales by 30%, reduced processing time by 2 hours per week. Numbers stand out to both ATS and human reviewers."},
    {"title": "Common ATS Rejection Reasons", "category": "ats_basics", "content": "The most common reasons CVs fail ATS screening are: missing keywords from the job description, unusual section headings the ATS cannot recognise, complex formatting that garbles the parsed text, and applying for roles where the experience level does not match."},
    {"title": "Cover Letter Best Practices", "category": "cover_letter", "content": "A strong cover letter opens with a specific reason you want this role at this company, not a generic introduction. The middle paragraph connects your top 2-3 achievements directly to the job requirements. Close with a clear call to action. Keep it to one page."},
    {"title": "Experience Level Matching", "category": "tailoring", "content": "Applying for roles above your experience level is a common reason for rejection. If a role requires 5 years of experience and you have 2, ATS may filter you out automatically. Focus applications on roles where you meet at least 70% of the requirements."},
    {"title": "Education and Certifications", "category": "keywords", "content": "List your highest qualification first. Include relevant certifications with the full name and abbreviation, for example Project Management Professional (PMP), so ATS can match either version. Include the awarding institution and year."},
    {"title": "File Format Recommendations", "category": "formatting", "content": "Submit CVs as .docx or PDF unless the application specifically requests otherwise. Avoid .pages, .odt, or image-based PDFs which many ATS systems cannot parse. If in doubt, .docx is the safest format for ATS compatibility."}
]

index = Index(
    text_fields=["title", "content"],
    keyword_fields=["category"]
)
index.fit(documents)
print("Index ready")

Index ready


In [8]:
def suggest_improvements(cv_text: str, missing_keywords: List[str]) -> List[str]:
    # search the knowledge base for relevant best practices
    search_results = index.search(
        f"keywords missing skills {' '.join(missing_keywords[:3])}",
        num_results=3
    )
    
    context = json.dumps(search_results, indent=2)
    
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "system",
                "content": """You are an ATS and CV expert. Generate specific, actionable 
CV improvement suggestions based on missing keywords and ATS best practices.

Rules:
- Be specific, not generic. Bad: "add more skills". Good: "Add a Skills section listing numpy, Tableau, and logistics data analysis"
- Each suggestion should be one concrete action the person can take
- Base suggestions on the provided ATS best practices context
- Return a JSON object with a single key "suggestions" containing a list of strings"""
            },
            {
                "role": "user",
                "content": f"""Missing keywords: {missing_keywords}

ATS Best Practices Context:
{context}

CV Text:
{cv_text}

Generate 4-5 specific improvement suggestions."""
            }
        ],
        response_format={"type": "json_object"}
    )
    
    result = json.loads(response.choices[0].message.content)
    return result["suggestions"]

In [9]:
suggestions = suggest_improvements(test_cv, score_result["missing_keywords"])
for i, s in enumerate(suggestions, 1):
    print(f"{i}. {s}")

1. Add a Skills section listing numpy, Tableau, and logistics data analysis to include missing technical keywords
2. Modify the Skills section to include the keyword 'logistics' to match job description terminology
3. Insert a bullet point under the Senior Data Analyst role mentioning 'supply chain data analysis' to incorporate a crucial missing keyword
4. Reformat the CV into a single-column layout and save as a plain .docx to comply with ATS formatting best practices
5. Include 'logistics data' as a separate skill in the Skills section to ensure all required keywords are directly listed


In [10]:
def generate_cover_letter(cv_text: str, job_description: str, match_score: int) -> str:
    
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "system",
                "content": """You are an expert cover letter writer. Write a tailored, 
professional cover letter based on the candidate's CV and the job description.

Rules:
- Open with a specific reason for applying to this role, not a generic intro
- Middle paragraph connects the candidate's top 2-3 achievements to the job requirements
- Acknowledge any gaps honestly if match score is below 70
- Close with a clear call to action
- Keep it to 3 paragraphs, under 250 words
- Do not use placeholder text like [Company Name] - infer from the job description"""
            },
            {
                "role": "user",
                "content": f"""CV:
{cv_text}

Job Description:
{job_description}

Match Score: {match_score}/100

Write a tailored cover letter."""
            }
        ]
    )
    
    return response.choices[0].message.content

In [11]:
cover_letter = generate_cover_letter(test_cv, test_jd, score_result["match_score"])
print(cover_letter)

I am excited to apply for the Data Analyst role at the Logistics Company, drawn by the opportunity to leverage my analytical skills to drive insights in the logistics industry. With a strong background in data analysis and a passion for working with complex data sets, I am confident that my skills align with the company's needs.

As a Senior Data Analyst at ABC Corp, I have developed strong SQL skills, creating queries to analyze customer data and inform business decisions. I have also utilized Python and pandas for data cleaning and analysis, and created interactive Power BI dashboards to present insights to stakeholders. Although I don't have direct experience in the logistics industry, I am eager to apply my skills to a new domain and learn from the company's expertise. I acknowledge that my experience may not perfectly match the job requirements, but I am excited to discuss how my transferable skills can contribute to the company's success.

I would welcome the opportunity to discu

In [12]:
import inspect
import json
from typing import List, get_type_hints

def get_instance_tools(instance) -> list:
    """
    Extract public methods from a class instance and convert them
    into Groq-compatible tool definitions automatically.
    """
    tools = []
    
    for name, method in inspect.getmembers(instance, predicate=inspect.ismethod):
        if name.startswith('_'):
            continue
            
        sig = inspect.signature(method)
        hints = get_type_hints(method)
        doc = inspect.getdoc(method) or ""
        
        properties = {}
        required = []
        
        for param_name, param in sig.parameters.items():
            if param_name == 'self':
                continue
            
            hint = hints.get(param_name, str)
            
            if hint == str:
                param_type = "string"
            elif hint == int:
                param_type = "integer"
            elif hint == List[str]:
                param_type = "array"
            elif hint == dict:
                param_type = "object"
            else:
                param_type = "string"
            
            prop = {"type": param_type, "description": param_name.replace('_', ' ')}
            if param_type == "array":
                prop["items"] = {"type": "string"}
                
            properties[param_name] = prop
            
            if param.default == inspect.Parameter.empty:
                required.append(param_name)
        
        tools.append({
            "type": "function",
            "function": {
                "name": name,
                "description": doc,
                "parameters": {
                    "type": "object",
                    "properties": properties,
                    "required": required
                }
            }
        })
    
    return tools

In [ ]:
class ATSTools:
    def __init__(self, client, index):
        self.client = client
        self.index = index
    
    def extract_job_requirements(self, job_description: str) -> dict:
        """Extracts structured requirements from a job description including required skills, experience level, and keywords"""
        schema = JobRequirements.model_json_schema()
        response = self.client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[
                {"role": "system", "content": f"Extract structured requirements from a job description. Return only a JSON object matching this schema: {schema}. Return only the JSON, no other text."},
                {"role": "user", "content": job_description}
            ],
            response_format={"type": "json_object"},
            temperature=0
        )
        return json.loads(response.choices[0].message.content)
    
    def score_cv(self, cv_text: str, requirements: dict) -> dict:
        """Scores a CV against job requirements and identifies matched and missing keywords"""
        schema = CVScore.model_json_schema()
        
        rubric = """
    Score the CV using this exact rubric:
    - Start at 100
    - Subtract 10 points for each required skill missing from the CV
    - Subtract 15 points if experience years in CV is less than required
    - Subtract 5 points for each important keyword missing
    - Minimum score is 0

    Be strict and literal — only count a skill as present if it is 
    explicitly mentioned in the CV text. Do not infer or assume.
    """
        
        response = self.client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[
                {
                    "role": "system",
                    "content": f"""You are an ATS scoring system. {rubric}
    Return only a JSON object matching this schema: {schema}
    Return only the JSON, no other text."""
                },
                {
                    "role": "user",
                    "content": f"CV:\n{cv_text}\n\nRequirements:\n{json.dumps(requirements, indent=2)}"
                }
            ],
            response_format={"type": "json_object"},
            temperature=0
        )
        return json.loads(response.choices[0].message.content)
    
    def suggest_improvements(self, cv_text: str, missing_keywords: List[str]) -> dict:
        """Generates specific actionable CV improvement suggestions based on missing keywords using ATS best practices"""
        search_results = self.index.search(
            f"keywords missing skills {' '.join(missing_keywords[:3])}",
            num_results=3
        )
        context = json.dumps(search_results, indent=2)
        response = self.client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[
                {"role": "system", "content": "Generate specific actionable CV improvement suggestions. Return a JSON object with a single key 'suggestions' containing a list of strings. Be specific not generic."},
                {"role": "user", "content": f"Missing keywords: {missing_keywords}\n\nATS Best Practices:\n{context}\n\nCV:\n{cv_text}\n\nGenerate 4-5 specific suggestions."}
            ],
            response_format={"type": "json_object"},
            temperature=0
        )
        return json.loads(response.choices[0].message.content)
    
    def generate_cover_letter(self, cv_text: str, job_description: str, match_score: int) -> str:
        """Generates a tailored cover letter based on the CV, job description and match score"""
        response = self.client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[
                {"role": "system", "content": "Write a tailored 3-paragraph cover letter. Open with a specific reason for applying. Connect top achievements to job requirements. Close with a call to action. Under 250 words."},
                {"role": "user", "content": f"CV:\n{cv_text}\n\nJob Description:\n{job_description}\n\nMatch Score: {match_score}/100"}
            ]
        )
        return response.choices[0].message.content

In [18]:
# instantiate tools
ats_tools = ATSTools(client, index)

# auto-generate tool definitions
tools = get_instance_tools(ats_tools)

# verify what was generated
for t in tools:
    print(f"Tool: {t['function']['name']} — {t['function']['description'][:60]}")

Tool: extract_job_requirements — Extracts structured requirements from a job description incl
Tool: generate_cover_letter — Generates a tailored cover letter based on the CV, job descr
Tool: score_cv — Scores a CV against job requirements and identifies matched 
Tool: suggest_improvements — Generates specific actionable CV improvement suggestions bas


In [19]:
def run_tool(name: str, args: dict):
    method = getattr(ats_tools, name, None)
    if method:
        return method(**args)
    return {"error": f"Unknown tool: {name}"}


instructions = """
You are an ATS Gap Analyser assistant. You help job seekers understand 
why their CV may not be passing ATS screening and what to fix.

When a user provides both a CV and a job description, always follow 
this sequence:
1. Call extract_job_requirements on the job description
2. Call score_cv with the CV and extracted requirements
3. Call suggest_improvements with the CV and missing keywords
4. Call generate_cover_letter with the CV, job description, and match score
5. Present a clear summary with the score, gaps, suggestions, and cover letter

Be specific and actionable. Never give generic advice.
"""


def run_agent(user_message: str):
    messages = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": user_message}
    ]
    
    print("Starting agent...\n")
    
    while True:
        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=messages,
            tools=tools,
        )
        
        choice = response.choices[0]
        messages.append(choice.message)
        
        if choice.finish_reason == "stop":
            return choice.message.content
        
        if choice.finish_reason == "tool_calls":
            for tool_call in choice.message.tool_calls:
                name = tool_call.function.name
                args = json.loads(tool_call.function.arguments)
                
                print(f"Calling tool: {name}")
                result = run_tool(name, args)
                
                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": json.dumps(result)
                })

In [21]:
answer = run_agent(f"""
Please analyse my CV against this job description and tell me:
1. How well does my CV match?
2. What are the gaps?
3. What should I fix?
4. Can you write me a cover letter?

CV:
{test_cv}

Job Description:
{test_jd}
""")

print(answer)

Starting agent...

Calling tool: extract_job_requirements
Calling tool: score_cv
Calling tool: suggest_improvements
Calling tool: generate_cover_letter
Your CV matches the job description at a score of 80. 
The gaps in your CV include:
- Lack of direct experience in logistics or supply chain data
- Missing technical skills such as dbt, Azure, and AWS
- No mention of freight industry knowledge

To improve your CV, consider adding the following:
- Include 'logistics' and 'supply chain' in your Skills section
- Add 'dbt' as a skill to demonstrate expertise in data transformation and loading
- Mention experience with cloud platforms such as Azure and AWS
- Emphasize industry relevance by adding 'Freight industry' to your Skills section
- Rephrase your Experience section to include keywords like 'data analysis for logistics' or 'supply chain optimization using SQL and Power BI'

Here is a cover letter tailored to the job description and your CV:
"I am excited to apply for the Data Analyst p